<a href="https://colab.research.google.com/github/TediBalint/AI-Jegyzetek/blob/master/Contrastive%20Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Contrastive Learning

Ebben a notebookban a **kontrasztív tanulás** módszereit vizsgáljuk.

## Tartalomjegyzék

1. Kontrastív tanulás alapjai
2. Pozitív és negatív párok
3. Kontrastív veszteségfüggvények
4. SimCLR framework
5. Reprezentáció tanulás

## 1. Kontrastív tanulás alapjai

### Alapötlet

| Koncepció | Leírás |
|-----------|--------|
| Cél | Hasonló minták közel, különbözők távol |
| Reprezentáció | Embedding térben |
| Felügyelet | Self-supervised (augmentáció) |

### Miért működik

1. **Pozitív párok**: Ugyanaz az adat, más nézet (augmentáció)
2. **Negatív párok**: Különböző adatok
3. **Tanulás**: Pozitívak közelítése, negatívak távolítása

### Alkalmazások

- Képosztályozás (SimCLR, MoCo)
- NLP (SimCSE)
- Beszédfelismerés

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

# Kontrastív tanulás illusztrálása
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Eredeti tér
X, y = make_blobs(n_samples=300, centers=3, cluster_std=2.0, random_state=42)
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', s=30)
axes[0].set_title('Eredeti tér (átfedő klaszterek)')

# Kontrastív tanulás hatása (szimulált)
# Szeparáltabb reprezentáció
X_sep, _ = make_blobs(n_samples=300, centers=3, cluster_std=0.5, random_state=42)
axes[1].scatter(X_sep[:, 0], X_sep[:, 1], c=y, cmap='viridis', s=30)
axes[1].set_title('Kontrastív reprezentáció (szeparált)')

# Pozitív/negatív párok
ax = axes[2]
anchor = np.array([0, 0])
positive = np.array([0.3, 0.2])
negatives = np.array([[2, 1], [-1, 2], [1.5, -1.5]])

ax.scatter(*anchor, c='blue', s=200, marker='*', label='Anchor', zorder=5)
ax.scatter(*positive, c='green', s=150, marker='o', label='Pozitív')
ax.scatter(negatives[:, 0], negatives[:, 1], c='red', s=150, marker='x', label='Negatív')

ax.annotate('', xy=positive, xytext=anchor,
            arrowprops=dict(arrowstyle='->', color='green', lw=2))
for neg in negatives:
    ax.annotate('', xy=neg, xytext=anchor,
                arrowprops=dict(arrowstyle='->', color='red', lw=1, ls='--'))

ax.set_title('Kontrastív tanulás: vonzás vs taszítás')
ax.legend()
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)

plt.tight_layout()
plt.show()

## 2. Pozitív és negatív párok

### Pár generálás

| Típus | Generálás | Példa |
|-------|-----------|-------|
| **Pozitív** | Augmentáció | Forgatás, vágás, zaj |
| **Negatív** | Más minták | Batch többi eleme |

### Augmentációk (képekhez)

- Random crop + resize
- Color jitter
- Gaussian blur
- Horizontal flip

In [ ]:
# Egyszerű augmentáció táblázatos adathoz

def augment_tabular(X, noise_std=0.1):
    """Egyszerű augmentáció: Gauss zaj hozzáadása."""
    noise = np.random.normal(0, noise_std, X.shape)
    return X + noise

def augment_dropout(X, dropout_rate=0.1):
    """Feature dropout augmentáció."""
    mask = np.random.random(X.shape) > dropout_rate
    return X * mask

# Demonstráció
X_demo = np.array([[1, 2, 3, 4, 5]])

print("Eredeti:", X_demo)
print("Zaj aug:", augment_tabular(X_demo, noise_std=0.2))
print("Dropout:", augment_dropout(X_demo, dropout_rate=0.3))

In [ ]:
# Pozitív/negatív pár vizualizáció

X, y = make_moons(n_samples=200, noise=0.1, random_state=42)

# Egy anchor kiválasztása
anchor_idx = 50
anchor = X[anchor_idx]
anchor_class = y[anchor_idx]

# Pozitív: augmentált változat
positives = np.array([augment_tabular(anchor.reshape(1, -1), noise_std=0.1).flatten()
                      for _ in range(5)])

# Negatív: más osztályból
negative_mask = y != anchor_class
negative_idx = np.random.choice(np.where(negative_mask)[0], 5, replace=False)
negatives = X[negative_idx]

fig, ax = plt.subplots(figsize=(10, 8))

# Háttér pontok
ax.scatter(X[:, 0], X[:, 1], c='lightgray', s=20, alpha=0.5)

# Anchor
ax.scatter(*anchor, c='blue', s=300, marker='*', label='Anchor', zorder=5)

# Pozitívak
ax.scatter(positives[:, 0], positives[:, 1], c='green', s=100, marker='o',
           label='Pozitív (augmentált)')

# Negatívak
ax.scatter(negatives[:, 0], negatives[:, 1], c='red', s=100, marker='x',
           label='Negatív (más osztály)')

ax.set_title('Pozitív és negatív párok')
ax.legend()
plt.show()

## 3. Kontrastív veszteségfüggvények

### NT-Xent (Normalized Temperature-scaled Cross Entropy)

$$\mathcal{L}_{i,j} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_{k=1}^{2N} \mathbf{1}_{[k \neq i]} \exp(\text{sim}(z_i, z_k) / \tau)}$$

ahol:
- $z_i, z_j$: pozitív pár reprezentációi
- $\tau$: hőmérséklet paraméter
- $\text{sim}$: koszinusz hasonlóság

### Triplet Loss

$$\mathcal{L} = \max(0, d(a, p) - d(a, n) + m)$$

ahol:
- $a$: anchor
- $p$: pozitív
- $n$: negatív
- $m$: margin

In [ ]:
# Kontrastív veszteségfüggvények implementációja

def cosine_similarity(a, b):
    """Koszinusz hasonlóság."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def nt_xent_loss(z_i, z_j, negatives, temperature=0.5):
    """
    NT-Xent loss egy pozitív párra.
    """
    sim_pos = cosine_similarity(z_i, z_j)

    # Negatívakkal hasonlóság
    sim_neg = [cosine_similarity(z_i, neg) for neg in negatives]

    # Softmax számláló és nevező
    numerator = np.exp(sim_pos / temperature)
    denominator = numerator + sum(np.exp(s / temperature) for s in sim_neg)

    loss = -np.log(numerator / denominator)
    return loss

def triplet_loss(anchor, positive, negative, margin=1.0):
    """
    Triplet loss.
    """
    d_pos = np.linalg.norm(anchor - positive)
    d_neg = np.linalg.norm(anchor - negative)
    return max(0, d_pos - d_neg + margin)

# Példa
anchor = np.array([1.0, 0.0, 0.0])
positive = np.array([0.9, 0.1, 0.0])  # Közeli
negative = np.array([-0.5, 0.5, 0.5])  # Távoli
other_negatives = [np.array([0.0, 1.0, 0.0]), np.array([0.0, 0.0, 1.0])]

print("NT-Xent loss:", nt_xent_loss(anchor, positive, other_negatives + [negative]))
print("Triplet loss:", triplet_loss(anchor, positive, negative))

In [ ]:
# Hőmérséklet hatása

temperatures = [0.1, 0.5, 1.0, 2.0]
similarities = np.linspace(-1, 1, 100)

fig, ax = plt.subplots(figsize=(10, 6))

for temp in temperatures:
    # Softmax a hasonlóságokra
    scaled = similarities / temp
    softmax = np.exp(scaled) / np.sum(np.exp(scaled))
    ax.plot(similarities, softmax, label=f'τ={temp}')

ax.set_xlabel('Koszinusz hasonlóság')
ax.set_ylabel('Softmax valószínűség')
ax.set_title('Hőmérséklet (τ) hatása')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print("τ hatása:")
print("- Alacsony τ: Éles eloszlás, hard negatives fontosabb")
print("- Magas τ: Lágy eloszlás, minden negatív számít")

## 4. SimCLR framework

### SimCLR architektúra

```
Kép → Augmentáció₁ → Encoder → Projection Head → z₁
      Augmentáció₂ → Encoder → Projection Head → z₂
                     ↓
              NT-Xent Loss(z₁, z₂)
```

### Komponensek

| Komponens | Szerep |
|-----------|--------|
| Encoder | Feature extractor (ResNet) |
| Projection head | MLP, reprezentáció térbe |
| Augmentáció | Pozitív pár generálás |

In [ ]:
# Egyszerűsített SimCLR PyTorch-ban

class SimpleEncoder(nn.Module):
    """Egyszerű encoder háló."""
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        # Projection head
        self.projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

    def forward(self, x):
        h = self.encoder(x)  # Reprezentáció
        z = self.projection(h)  # Projekció
        return h, F.normalize(z, dim=-1)  # L2 normalizált

def nt_xent_loss_torch(z1, z2, temperature=0.5):
    """
    NT-Xent loss PyTorch-ban.
    """
    batch_size = z1.shape[0]

    # Összes pár hasonlósága
    z = torch.cat([z1, z2], dim=0)  # [2B, D]
    sim = torch.mm(z, z.t()) / temperature  # [2B, 2B]

    # Maszk: diagonális és pozitív párok
    sim_i_j = torch.diag(sim, batch_size)  # Pozitív párok
    sim_j_i = torch.diag(sim, -batch_size)

    positive = torch.cat([sim_i_j, sim_j_i], dim=0)

    # Negatív maszk (nem önmaga és nem a pozitív pár)
    mask = (~torch.eye(2 * batch_size, dtype=torch.bool)).float()

    # Loss
    numerator = torch.exp(positive)
    denominator = (mask * torch.exp(sim)).sum(dim=1)

    loss = -torch.log(numerator / denominator).mean()
    return loss

# Teszt
encoder = SimpleEncoder(input_dim=2, hidden_dim=64, output_dim=32)
x = torch.randn(16, 2)
_, z = encoder(x)
print(f"Reprezentáció alakja: {z.shape}")
print(f"L2 norma: {z.norm(dim=1).mean():.4f}")

In [ ]:
# SimCLR tanítás

def train_simclr(X, encoder, epochs=100, lr=0.01, temperature=0.5, noise_std=0.1):
    """
    Egyszerű SimCLR tanítás.
    """
    optimizer = torch.optim.Adam(encoder.parameters(), lr=lr)
    X_tensor = torch.FloatTensor(X)

    losses = []

    for epoch in range(epochs):
        # Augmentált változatok
        noise1 = torch.randn_like(X_tensor) * noise_std
        noise2 = torch.randn_like(X_tensor) * noise_std
        x1 = X_tensor + noise1
        x2 = X_tensor + noise2

        # Forward
        _, z1 = encoder(x1)
        _, z2 = encoder(x2)

        # Loss
        loss = nt_xent_loss_torch(z1, z2, temperature)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}")

    return losses

# Adat
X, y = make_moons(n_samples=500, noise=0.1, random_state=42)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Tanítás
encoder = SimpleEncoder(input_dim=2, hidden_dim=64, output_dim=32)
losses = train_simclr(X_scaled, encoder, epochs=100)

In [ ]:
# Tanulási görbe és reprezentáció

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Loss
axes[0].plot(losses)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('NT-Xent Loss')
axes[0].set_title('Tanulási görbe')
axes[0].grid(True, alpha=0.3)

# Eredeti tér
axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y, cmap='coolwarm', s=20)
axes[1].set_title('Eredeti tér')

# Tanult reprezentáció
with torch.no_grad():
    h, z = encoder(torch.FloatTensor(X_scaled))
    z_np = z.numpy()

# t-SNE a reprezentációra
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
z_2d = tsne.fit_transform(z_np)

axes[2].scatter(z_2d[:, 0], z_2d[:, 1], c=y, cmap='coolwarm', s=20)
axes[2].set_title('SimCLR reprezentáció (t-SNE)')

plt.tight_layout()
plt.show()

## 5. Reprezentáció tanulás

### Downstream task

A kontrastív tanulás után a reprezentáció használható:

| Task | Módszer |
|------|--------|
| Osztályozás | Linear probing |
| Fine-tuning | Teljes modell hangolás |
| Transfer | Más domain |

In [ ]:
# Linear probing: osztályozó a tanult reprezentációra

from sklearn.model_selection import train_test_split

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42
)

# Reprezentációk
with torch.no_grad():
    h_train, _ = encoder(torch.FloatTensor(X_train))
    h_test, _ = encoder(torch.FloatTensor(X_test))
    h_train = h_train.numpy()
    h_test = h_test.numpy()

# Összehasonlítás: eredeti vs reprezentáció

# Különböző címkézett arányok
label_ratios = [0.01, 0.05, 0.1, 0.2, 0.5, 1.0]
results_raw = []
results_repr = []

for ratio in label_ratios:
    n_labeled = max(2, int(len(X_train) * ratio))
    idx = np.random.choice(len(X_train), n_labeled, replace=False)

    # Eredeti feature-ökön
    clf_raw = LogisticRegression(random_state=42)
    clf_raw.fit(X_train[idx], y_train[idx])
    acc_raw = accuracy_score(y_test, clf_raw.predict(X_test))
    results_raw.append(acc_raw)

    # Tanult reprezentáción
    clf_repr = LogisticRegression(random_state=42)
    clf_repr.fit(h_train[idx], y_train[idx])
    acc_repr = accuracy_score(y_test, clf_repr.predict(h_test))
    results_repr.append(acc_repr)

# Vizualizáció
plt.figure(figsize=(10, 6))
plt.plot(label_ratios, results_raw, 'ro-', linewidth=2, markersize=10, label='Eredeti feature-ök')
plt.plot(label_ratios, results_repr, 'bs-', linewidth=2, markersize=10, label='SimCLR reprezentáció')
plt.xlabel('Címkézett adat aránya')
plt.ylabel('Test accuracy')
plt.title('Linear Probing: Eredeti vs Kontrastív reprezentáció')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("\nEredmények:")
for ratio, raw, repr_ in zip(label_ratios, results_raw, results_repr):
    print(f"  {ratio*100:5.1f}%: Eredeti={raw:.3f}, SimCLR={repr_:.3f}")

In [ ]:
# MoCo vs SimCLR összehasonlítás (konceptuális)

comparison_data = {
    'Jellemző': ['Negatív minták', 'Momentum encoder', 'Memory bank', 'Batch méret'],
    'SimCLR': ['Batch-ből', 'Nincs', 'Nincs', 'Nagy (4096+)'],
    'MoCo': ['Queue-ból', 'Van', 'Van', 'Kis (256)']
}

fig, ax = plt.subplots(figsize=(10, 4))
ax.axis('off')

table = ax.table(
    cellText=[comparison_data['SimCLR'], comparison_data['MoCo']],
    rowLabels=['SimCLR', 'MoCo'],
    colLabels=comparison_data['Jellemző'],
    loc='center',
    cellLoc='center'
)
table.scale(1.2, 2)
table.auto_set_font_size(False)
table.set_fontsize(11)

plt.title('SimCLR vs MoCo összehasonlítás', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

## Összefoglalás

### Kontrastív tanulás:

| Komponens | Leírás |
|-----------|--------|
| Pozitív pár | Ugyanaz az adat, más augmentáció |
| Negatív pár | Különböző adatok |
| Cél | Pozitívak közel, negatívak távol |

### NT-Xent loss:

$$\mathcal{L} = -\log \frac{\exp(\text{sim}(z_i, z_j) / \tau)}{\sum_k \exp(\text{sim}(z_i, z_k) / \tau)}$$

### Főbb módszerek:

| Módszer | Jellemző |
|---------|----------|
| SimCLR | Nagy batch, egyszerű |
| MoCo | Memory bank, momentum |
| BYOL | Negatív nélkül |

### Alkalmazás:

- Self-supervised pretraining
- Kevés címkézett adat
- Transfer learning